In [ ]:
import numpy as np
import pandas as pd
import ast

# --- PATHS --- (UPDATE THESE TO YOUR LOCAL PATHS)
ML25M_MOVIES_PATH = "movies.csv"
ML25M_LINKS_PATH = "links.csv"
TMDB_MOVIES_PATH = "tmdb_5000_movies.csv"
TMDB_CREDITS_PATH = "tmdb_5000_credits.csv"

print("Loading datasets...")
# 1. Load MovieLens 25M
ml_movies = pd.read_csv(ML25M_MOVIES_PATH)
links = pd.read_csv(ML25M_LINKS_PATH)
ml_movies = ml_movies.merge(links, on='movieId')
ml_movies.dropna(subset=['tmdbId'], inplace=True)
ml_movies['tmdbId'] = ml_movies['tmdbId'].astype(int)

# 2. Load TMDB metadata
tmdb_movies = pd.read_csv(TMDB_MOVIES_PATH)
tmdb_credits = pd.read_csv(TMDB_CREDITS_PATH)
tmdb = tmdb_movies.merge(tmdb_credits, left_on='id', right_on='movie_id')

# 3. Merge MovieLens with TMDB on tmdbId
movies = ml_movies.merge(tmdb, left_on='tmdbId', right_on='id', how='inner')

# Keep relevant columns (Using MovieLens title and genres, TMDB overview and cast)
movies = movies[['tmdbId', 'title_x', 'genres_x', 'overview', 'cast', 'crew', 'vote_average']]
movies.rename(columns={'title_x': 'title', 'genres_x': 'genres', 'tmdbId': 'id'}, inplace=True)
movies.dropna(subset=['overview'], inplace=True)
movies.drop_duplicates(subset=['id'], inplace=True)
print(f"Merged Dataset Size: {movies.shape[0]} movies")


In [ ]:
def convert_cast(text):
    if not isinstance(text, str): return []
    l = []
    counter = 0
    for i in ast.literal_eval(text):
        if counter < 4:
            l.append(i['name'])
            counter += 1
        else:
            break
    return l

def convert_crew(text):
    if not isinstance(text, str): return []
    l = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            l.append(i['name'])
            break
    return l

# MovieLens genres are formatted as 'Action|Adventure|Sci-Fi'
movies['genres'] = movies['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])
movies['cast'] = movies['cast'].apply(convert_cast)
movies['crew'] = movies['crew'].apply(convert_crew)


In [ ]:
def clean(text):
    return [i.replace(" ", "").lower() for i in text]

movies['genres_clean'] = movies['genres'].apply(clean)
movies['crew_clean'] = movies['crew'].apply(clean)
movies['cast_clean'] = movies['cast'].apply(clean)


In [ ]:
# Convert lists back to space-separated strings for tags
genres_str = movies['genres_clean'].apply(lambda x: ' '.join(x))
cast_str = movies['cast_clean'].apply(lambda x: ' '.join(x))
crew_str = movies['crew_clean'].apply(lambda x: ' '.join(x))
overview_str = movies['overview'].apply(lambda x: str(x).lower())

# Combine for semantic embeddings
movies['tags'] = (
    movies['title'].str.lower() + ' '
    + overview_str + ' '
    + genres_str + ' '
    + cast_str + ' '
    + crew_str
)

# Convert display versions of lists to comma separated strings
movies['cast_display'] = movies['cast'].apply(lambda x: ', '.join(x))
movies['genres_display'] = movies['genres'].apply(lambda x: ', '.join(x))

new_df = movies[['id', 'title', 'tags', 'overview', 'cast_display', 'genres_display', 'vote_average']].copy()
new_df.reset_index(drop=True, inplace=True)


In [ ]:
!pip install sentence-transformers faiss-cpu
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

print("Loading high-accuracy SentenceTransformer model (all-mpnet-base-v2)...")
model = SentenceTransformer('all-mpnet-base-v2')

print("Generating embeddings... (This will take a significant amount of time for 62k+ movies)")
embeddings = model.encode(new_df['tags'].tolist(), show_progress_bar=True)


In [ ]:
print("Building FAISS index...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
faiss.normalize_L2(embeddings)
index.add(embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")


In [ ]:
import pickle
import os

os.makedirs("backend", exist_ok=True)

print("Saving DataFrame and FAISS index...")
# Save them into the backend directory for easy access by FastAPI
pickle.dump(new_df, open('backend/movies.pkl','wb'))
faiss.write_index(index, 'backend/movies.index')
print("Saved successfully!")
